In [1]:
import numpy as np
data = np.arange(12).reshape(3,4)
obj = np.exp(data) #广播运算，将data每一个值取自然对数
obj

array([[1.00000000e+00, 2.71828183e+00, 7.38905610e+00, 2.00855369e+01],
       [5.45981500e+01, 1.48413159e+02, 4.03428793e+02, 1.09663316e+03],
       [2.98095799e+03, 8.10308393e+03, 2.20264658e+04, 5.98741417e+04]])

In [2]:
# generate_jv_data.py
import numpy as np
import pandas as pd
import os

os.makedirs("output", exist_ok=True)
#创建"output"文件夹方法，如果文件夹已存在，不会报错、直接忽略

def single_diode_jv(V, Jsc, Voc, n=1.8, Rs=0.5, Rsh=1000): 
    '''电压、短路电流密度、开路电压、理想因子、串联电阻、并联电阻'''
    # 单二极管模型，生成J-V，正向扫描
    q = 1.602e-19
    k = 1.38e-23
    T = 298
    Vt = k*T/q #热电压，常温下约25.7mV
    J0 = Jsc / (np.exp(Voc / (n * Vt)) - 1.0)   # 按 Voc 反推饱和电流
    J  = Jsc - J0 * (np.exp(V/(n*Vt)) - 1.0) - V/Rsh
    return J #返回numpy数组

curve_list = []
np.random.seed(42)
n_sample = 20  # 一共20条电池样品，后面画效率分布够用

for sample_id in range(n_sample):
    # 随机一组合理参数，模拟不同电池
    Jsc_true = np.random.uniform(22, 26) #在 [22,26) 区间生成一个浮点数
    Voc_true = np.random.uniform(1.0, 1.15)
    V_scan = np.linspace(-0.1, Voc_true+0.1, 60)  # 电压扫描范围
    J_scan = single_diode_jv(V_scan, Jsc_true, Voc_true)
    for v, j in zip(V_scan, J_scan):
        curve_list.append({
            "sample_id": sample_id,
            "V": v,
            "J": j
        })

df = pd.DataFrame(curve_list)
df.to_csv("output/jv_data.csv", index=False)
print("✅ 模拟J-V数据已经生成到 output/jv_data.csv")
print("字段：sample_id（样品编号）, V(电压V), J(电流密度 mA/cm²)")

✅ 模拟J-V数据已经生成到 output/jv_data.csv
字段：sample_id（样品编号）, V(电压V), J(电流密度 mA/cm²)


In [3]:
obj1 = pd.read_csv("output/jv_data.csv")
# obj1 = obj1[obj1["J"]<0]
obj1

,sample_id,V,J
0,0,-0.100000,23.498260
1,0,-0.077244,23.498238
2,0,-0.054488,23.498215
3,0,-0.031732,23.498192
4,0,-0.008976,23.498169
...,...,...,...
1195,19,1.080191,-8.877319
1196,19,1.101649,-28.744259
1197,19,1.123107,-60.353454
1198,19,1.144565,-110.645112


In [7]:
import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline
from scipy.optimize import brentq

result = []
data = pd.read_csv("output/jv_data.csv")

for i_d, group in data.groupby("sample_id"):
    v_arr = group["V"].values
    j_arr = group["J"].values
    idx_sort = np.argsort(v_arr)
    v_arr, j_arr = v_arr[idx_sort], j_arr[idx_sort]

    cs = CubicSpline(v_arr, j_arr)
    Jsc = float(cs(0.0)) # V=0 处的电流密度

    idx = np.where(np.diff(np.sign(j_arr)) != 0)[0] # J 由正变负的位置
    if len(idx) == 0:
        raise ValueError(f"样品 {i_d} 的曲线没有穿过 J=0")
    i = int(idx[0])
    Voc = brentq(cs, v_arr[i], v_arr[i + 1]) # 在变号区间内求根

    v_sample = np.linspace(0.0, Voc, 1000) # 只在 [0, Voc] 上找 MPP
    p_sample = v_sample * cs(v_sample)
    Pmax = float(np.max(p_sample))

    result.append({
        "sample_id": i_d,
        "Jsc": Jsc,
        "Voc": Voc,
        "Pmax": Pmax,
        "FF": Pmax / (Voc * Jsc),
        "PCE": Pmax / 100.0 * 100,    # 100 mW/cm² 入射
    })

df_out = pd.DataFrame(result)
df_out.to_excel("output/result_summary.xlsx", index=False)
df_out.to_csv("output/result_summary.csv", index=False)
print("✅ 参数结果保存完成")
df_out.head(3).round(3)

✅ 参数结果保存完成


,sample_id,Jsc,Voc,Pmax,FF,PCE
0,0,23.498,1.143,22.425,0.835,22.425
1,1,24.928,1.090,22.532,0.829,22.532
2,2,22.624,1.023,19.020,0.821,19.020
